# Tool Messages

The `tool.py` module defines messages used to return tool-execution results to chat models. It also provides tool-call data structures, streaming tool-call chunks, validation helpers, and best-effort parsers for raw provider tool-call data.

# ToolOutputMixin

`ToolOutputMixin` marks objects that tools may return directly.

When a custom tool is invoked with a `ToolCall`, outputs that do not inherit from `ToolOutputMixin` are automatically converted to strings and wrapped in a `ToolMessage`.

# ToolMessage

`ToolMessage` represents the result of a tool execution that is returned to a chat model.

The `tool_call_id` field connects the tool result to the corresponding tool-call request. This is especially important when a model requests multiple tools in parallel.

## Bases

- `BaseMessage`
- `ToolOutputMixin`

## Attributes

1. `tool_call_id`: Stores the identifier of the tool call to which this message responds.
   * **Type:**
     ```python
     tool_call_id: str
     ```

2. `type`: Stores the message type used during serialization and deserialization.
   * **Type:**
     ```python
     type: Literal["tool"] = "tool"
     ```

3. `artifact`: Stores the complete tool output or additional data that should not be sent directly to the model.
   * **Type:**
     ```python
     artifact: Any = None
     ```

4. `status`: Stores whether the tool execution succeeded or failed.
   * **Type:**
     ```python
     status: Literal["success", "error"] = "success"
     ```

5. `additional_kwargs`: Stores additional message data inherited from `BaseMessage`.

   This field is currently not used by `ToolMessage`.

   * **Type:**
     ```python
     additional_kwargs: dict[Any, Any] = Field(
         default_factory=dict,
         repr=False
     )
     ```

6. `response_metadata`: Stores response metadata inherited from `BaseMessage`.

   This field is currently not used by `ToolMessage`.

   * **Type:**
     ```python
     response_metadata: dict[Any, Any] = Field(
         default_factory=dict,
         repr=False
     )
     ```

### Validators

1. `coerce_args`: Converts message content and the tool-call identifier into supported types.

   Tuple content is converted to a list. Non-string content values and unsupported list elements are converted to strings when possible. Numeric and UUID tool-call identifiers are converted to strings.

   * **Syntax:**
     ```python
     @model_validator(mode="before")
     @classmethod
     coerce_args(
         cls,
         values: dict[str, Any] # ToolMessage values to validate and normalize
     ) -> dict[str, Any]
     ```

### Methods

1. `__init__`: Initializes a tool message using raw content or standardized content blocks.
   * **Syntax:**
     ```python
     __init__(
         self,
         content: str | list[str | dict[Any, Any]] | None = None, # Tool-result content sent to the model
         content_blocks: list[types.ContentBlock] | None = None, # Standardized content blocks
         **kwargs: Any # Additional ToolMessage fields
     ) -> None
     ```

# ToolMessageChunk

`ToolMessageChunk` represents a partial tool-result message produced during streaming.

Compatible chunks can be combined while preserving the tool-call identifier and merging their content, artifact, metadata, and execution status.

## Bases

- `ToolMessage`
- `BaseMessageChunk`

## Attributes

1. `type`: Stores the chunk-specific message type used during serialization and deserialization.
   * **Type:**
     ```python
     type: Literal["ToolMessageChunk"] = "ToolMessageChunk"
     ```

### Methods

1. `__add__`: Combines the current tool-message chunk with another message chunk.

   When combining two `ToolMessageChunk` objects, their `tool_call_id` values must match. Otherwise, a `ValueError` is raised. If either chunk has an error status, the resulting chunk also has an error status.

   * **Syntax:**
     ```python
     __add__(
         self,
         other: Any # Message chunk or another supported value to combine
     ) -> BaseMessageChunk
     ```

# ToolCall

`ToolCall` is a `TypedDict` representing an AI model's request to execute a tool.

## Bases

- `TypedDict`

### Fields

1. `name`: Stores the name of the tool to invoke.
   * **Type:**
     ```python
     name: str
     ```

2. `args`: Stores the arguments passed to the tool.
   * **Type:**
     ```python
     args: dict[str, Any]
     ```

3. `id`: Stores an identifier used to associate the tool call with its result.
   * **Type:**
     ```python
     id: str | None
     ```

4. `type`: Stores the optional discriminator for the tool-call structure.
   * **Type:**
     ```python
     type: NotRequired[Literal["tool_call"]]
     ```

# ToolCallChunk

`ToolCallChunk` is a `TypedDict` representing a partial tool call produced during streaming.

When chunks are merged, their string fields are concatenated. Chunks are merged only when their non-null `index` values match.

## Bases

- `TypedDict`

### Fields

1. `name`: Stores a partial or complete tool name.
   * **Type:**
     ```python
     name: str | None
     ```

2. `args`: Stores partial tool arguments as a JSON-compatible string.
   * **Type:**
     ```python
     args: str | None
     ```

3. `id`: Stores the tool-call identifier.
   * **Type:**
     ```python
     id: str | None
     ```

4. `index`: Stores the position of the tool call in a sequence and is used when merging chunks.
   * **Type:**
     ```python
     index: int | None
     ```

5. `type`: Stores the optional discriminator for the tool-call-chunk structure.
   * **Type:**
     ```python
     type: NotRequired[Literal["tool_call_chunk"]]
     ```

## Functions

1. `tool_call`: Creates a standardized `ToolCall`.
   * **Syntax:**
     ```python
     tool_call(
         *,
         name: str, # Name of the tool to invoke
         args: dict[str, Any], # Arguments passed to the tool
         id: str | None # Identifier associated with the tool call
     ) -> ToolCall
     ```

2. `tool_call_chunk`: Creates a standardized `ToolCallChunk`.
   * **Syntax:**
     ```python
     tool_call_chunk(
         *,
         name: str | None = None, # Partial or complete tool name
         args: str | None = None, # Partial JSON argument string
         id: str | None = None, # Tool-call identifier
         index: int | None = None # Position of the tool call in a sequence
     ) -> ToolCallChunk
     ```

3. `invalid_tool_call`: Creates an `InvalidToolCall` for a tool call that could not be parsed.
   * **Syntax:**
     ```python
     invalid_tool_call(
         *,
         name: str | None = None, # Tool name when available
         args: str | None = None, # Unparsed tool arguments
         id: str | None = None, # Tool-call identifier
         error: str | None = None # Parsing or validation error
     ) -> InvalidToolCall
     ```

4. `default_tool_parser`: Performs best-effort parsing of raw tool-call dictionaries.

   Successfully decoded JSON arguments produce `ToolCall` objects. Tool calls with invalid JSON arguments produce `InvalidToolCall` objects. Entries without a `function` field are ignored.

   * **Syntax:**
     ```python
     default_tool_parser(
         raw_tool_calls: list[dict[str, Any]] # Raw tool-call dictionaries to parse
     ) -> tuple[list[ToolCall], list[InvalidToolCall]]
     ```

5. `default_tool_chunk_parser`: Performs best-effort parsing of raw streaming tool-call dictionaries.

   It extracts the function name, argument string, identifier, and index from each raw tool-call chunk.

   * **Syntax:**
     ```python
     default_tool_chunk_parser(
         raw_tool_calls: list[dict[str, Any]] # Raw streaming tool-call dictionaries
     ) -> list[ToolCallChunk]
     ```